# Credit Card Fraud Detection - Random Forest

This notebook trains a Random Forest Classifier using two imbalance handling techniques:
1. Random Undersampling
2. SMOTE (Synthetic Minority Over-sampling Technique)

We evaluate and compare their performance using metrics suitable for imbalanced datasets.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Load the dataset
df = pd.read_csv('creditcard.csv')
X = df.drop('Class', axis=1)
y = df['Class']

In [3]:
# Scale 'Time' and 'Amount' features
scaler = RobustScaler()
X_scaled = X.copy()
X_scaled[['Time', 'Amount']] = scaler.fit_transform(X[['Time', 'Amount']])

In [4]:
# Split data into train and test sets (stratified to maintain class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

Train size: (227845, 30), Test size: (56962, 30)


In [5]:
# Apply Random Undersampling to the training set only
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)
print(f"Undersampled train size: {X_train_under.shape}")

Undersampled train size: (788, 30)


In [6]:
# Train Random Forest model on undersampled data
# Use n_jobs=-1 for multi-core processing
rf_under = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_under.fit(X_train_under, y_train_under)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [7]:
# Predict and evaluate the undersampled Random Forest model
y_pred_under = rf_under.predict(X_test)
y_prob_under = rf_under.predict_proba(X_test)[:, 1]

print("--- Random Forest with Undersampling Performance ---")
print(classification_report(y_test, y_pred_under))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_under))

roc_auc = roc_auc_score(y_test, y_prob_under)
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob_under)
pr_auc = auc(recall_vals, precision_vals)
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")

--- Random Forest with Undersampling Performance ---
              precision    recall  f1-score   support

           0       1.00      0.96      0.98     56864
           1       0.04      0.92      0.08        98

    accuracy                           0.96     56962
   macro avg       0.52      0.94      0.53     56962
weighted avg       1.00      0.96      0.98     56962

Confusion Matrix:
[[54827  2037]
 [    8    90]]
ROC-AUC: 0.9777
PR-AUC: 0.7538


In [8]:
# Apply SMOTE to the training set only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(f"SMOTE resampled train size: {X_train_smote.shape}")

SMOTE resampled train size: (454902, 30)


In [9]:
# Train Random Forest model on SMOTE data
# Use max_depth=10 to speed up training on the 450k+ oversampled samples and prevent overfitting
rf_smote = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf_smote.fit(X_train_smote, y_train_smote)

,n_estimators,100
,criterion,'gini'
,max_depth,12
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
# Predict and evaluate the SMOTE Random Forest model
y_pred_smote = rf_smote.predict(X_test)
y_prob_smote = rf_smote.predict_proba(X_test)[:, 1]

print("--- Random Forest with SMOTE Performance ---")
print(classification_report(y_test, y_pred_smote))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_smote))

roc_auc_s = roc_auc_score(y_test, y_prob_smote)
precision_vals_s, recall_vals_s, _ = precision_recall_curve(y_test, y_prob_smote)
pr_auc_s = auc(recall_vals_s, precision_vals_s)
print(f"ROC-AUC: {roc_auc_s:.4f}")
print(f"PR-AUC: {pr_auc_s:.4f}")

--- Random Forest with SMOTE Performance ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.52      0.86      0.64        98

    accuracy                           1.00     56962
   macro avg       0.76      0.93      0.82     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix:
[[56785    79]
 [   14    84]]
ROC-AUC: 0.9777
PR-AUC: 0.8245


In [11]:
# Summary and Comparison
from sklearn.metrics import precision_recall_fscore_support
p_u, r_u, f_u, _ = precision_recall_fscore_support(y_test, y_pred_under, average=None)
p_s, r_s, f_s, _ = precision_recall_fscore_support(y_test, y_pred_smote, average=None)

summary_df = pd.DataFrame({
    'Metric': ['ROC-AUC', 'PR-AUC', 'Precision (Class 1)', 'Recall (Class 1)', 'F1-score (Class 1)'],
    'Undersampling': [roc_auc, pr_auc, p_u[1], r_u[1], f_u[1]],
    'SMOTE': [roc_auc_s, pr_auc_s, p_s[1], r_s[1], f_s[1]]
})
print("Model Comparison Summary:")
summary_df

Model Comparison Summary:


,Metric,Undersampling,SMOTE
0,ROC-AUC,0.977698,0.977669
1,PR-AUC,0.753826,0.824534
2,Precision (Class 1),0.042313,0.515337
3,Recall (Class 1),0.918367,0.857143
4,F1-score (Class 1),0.080899,0.643678
